# Beam modulation

`slow_down` gives the steady state of a beam that has been on for a long time. In experiments the beam power changes: it is switched on and off, fired in short blips to measure the slowing down, or modulated to measure the heating response. With fixed plasma profiles the fast ions respond linearly: each one slows down on its own, $\mathrm{d}v/\mathrm{d}t = -(v^3 + v_c^3) / (\tau_s v^2)$ (Stix, Plasma Phys. 14, 367, 1972), so a moment $Q$ of the fast ions (their number, energy, current, the heating of electrons and ions) at time $t$ is the source power convolved with the response of one birth at age $\tau$:

$$
Q(t) = \int_0^\infty s(t - \tau)\, K_Q(\tau)\, \mathrm{d}\tau, \qquad s = P(t) / P_\mathrm{ref}.
$$

`slowing_down_response(deposition, plasma, surfaces)` builds the kernels $K_Q$ of every shell from the births of a deposition, along the same trajectories as `slow_down` (drags, Coulomb logarithms, pitch modes of the current, end energy), and `response.evolve(time, power)` convolves them with any power waveform.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

import freeofcharge as foc
from freeofcharge.time_dependent import slowing_down_response

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs
plasma_plots.set_math("latex")  # formulas in the figures typeset as LaTeX

equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
plasma = foc.Plasma.parabolic(1e20, 2e4, ions={"D": (2, 1, 0.96), "He": (4, 2, 0.02)})
rho_edges = np.linspace(0.0, 1.0, 21)
surfaces = equilibrium.flux_surfaces(rho_edges)
origin, direction = foc.tangential_injection(5.3, 12.0, sense=1)
beam = foc.NeutralBeam.gaussian(
    origin, direction, 1.0e6, 16.5e6, width=0.3, divergence=0.005, markers=500, seed=1
)
deposition = foc.deposit(
    beam,
    equilibrium,
    plasma,
    step=0.03,
    births_per_marker=2,
    rho_edges=rho_edges,
    seed=2,
)
power = deposition.injected_power  # 16.5 MW

# the fast ions join the thermal ions at 1.5 T_i
steady = foc.slow_down(deposition, plasma, surfaces, thermal_factor=1.5)
response = slowing_down_response(deposition, plasma, surfaces, thermal_factor=1.5)
assert abs(response.steady_state[1].sum() / steady.stored_energy - 1) < 1e-6
print(
    f"steady state: fast-ion energy {steady.stored_energy / 1e6:.2f} MJ, "
    f"driven current {steady.driven_current / 1e6:.2f} MA"
)
print(
    f"slowing-down time on the electrons: {response.slowing_down_time[0]:.2f} s at the centre, "
    f"{response.slowing_down_time[-1]:.2f} s at the edge"
)

The response is that of the ITER-like tokamak of the guide "Heating and current drive", a 1 MeV deuterium beam of 16.5 MW. Its steady state is that of `slow_down` (to $10^{-6}$). The slowing-down time on the electrons $\tau_s$ falls from about 2 s at the centre to 0.15 s at the edge, with the electron temperature.

## Switch-on and switch-off

`evolve` takes the power on an equally spaced time grid, as a function of time or as one value per step, held constant over each step (where the convolution is exact). `reference_power` is the power at which the deposition was computed: with the injected power the waveform is an injected power. Before the first time the power is `initial_power` (zero here). The beam runs for 1.2 s:

In [ ]:
dt = 1e-3
time = np.arange(0.0, 2.5, dt)
on_off = response.evolve(
    time, lambda t: np.where(t < 1.2, power, 0.0), reference_power=power
)
before = on_off.sel(time=1.199)
assert abs(float(before.stored_energy) / steady.stored_energy - 1) < 5e-3
assert abs(float(before.driven_current) / steady.driven_current - 1) < 5e-3

traces = xr.Dataset(
    {
        "beam power": on_off.power / power,
        "fast-ion energy": on_off.stored_energy / steady.stored_energy,
        "driven current": on_off.driven_current / steady.driven_current,
    }
).to_array("quantity", name="relative to the steady state")
traces.attrs["units"] = "1"
traces.plasma.plot.profiles(x="time", over="quantity")

The fast-ion energy and the current reach the steady state after 1 s, the longest slowing down; after the switch-off they decay in the same way, since the response is linear: the decay is the steady state minus the rise. Per shell, the time to reach $1 - 1/e$ of the steady value (equal to the $1/e$ decay time after switch-off) follows the slowing down:

In [ ]:
def rise_time(values, final):
    with np.errstate(invalid="ignore", divide="ignore"):
        reached = values / final >= 1.0 - np.exp(-1.0)
    times = np.where(reached.any(axis=0), time[np.argmax(reached, axis=0)], np.nan)
    return np.where(np.abs(final) > 0.0, times, np.nan)


rising = on_off.sel(time=slice(0.0, 1.199))
times = xr.Dataset(
    {
        "fast-ion energy": (
            "rho",
            rise_time(rising.fast_ion_energy.values, rising.fast_ion_energy[-1].values),
        ),
        "driven current": (
            "rho",
            rise_time(rising.current_density.values, rising.current_density[-1].values),
        ),
        "thermalisation": on_off.thermalisation_time,
    },
    coords={"rho": on_off.rho},
).to_array("time", name="characteristic time")
times.attrs["units"] = "s"
times.plasma.plot.profiles(x="rho", over="time").show()
centre = times.isel(rho=0)
print(
    "at the centre: energy {:.2f} s, current {:.2f} s, thermalisation {:.2f} s".format(
        *centre.values
    )
)

The energy and the current take about 0.4 s to rise at the centre, under 0.1 s near the edge, and the time to slow down from 1 MeV to $1.5\,T_i$ falls from 0.95 s to 0.3 s. Both rise times are well below the thermalisation time: most of the energy and current is carried while the ions are fast. In the core the current rises a little later than the energy, further out a little sooner, where it is carried by younger ions. In the outer two shells there is no current: all births there are on trapped orbits.

## A 10 ms blip

A short pulse of the beam, as fired in blip experiments to measure the slowing down: the full 16.5 MW for 10 ms.

In [ ]:
dt = 5e-4
time = np.arange(0.0, 1.2, dt)
blip = response.evolve(
    time,
    lambda t: np.where((t >= 0.05) & (t < 0.06), power, 0.0),
    reference_power=power,
)
heating = blip.total_heating / 1e3
heating.attrs.update(units="kW", long_name="heating")
heating.plasma.plot.profiles(x="time", over="species").show()

# the shine-through takes its share of the injected energy
injected = float(blip.power.sum()) * dt
deposited = injected * response.birth_power / power
to_species = blip.total_heating.sum("time").values * dt
thermalised = float(blip.thermalised_power.sum()) * dt
assert abs((to_species.sum() + thermalised) / deposited - 1) < 1e-9
electrons, ions = blip.total_heating.values
crossing = time[(time > 0.06) & (ions > electrons)][0]
print(
    f"injected {injected / 1e3:.0f} kJ, deposited {deposited / 1e3:.1f} kJ: to the electrons {to_species[0] / 1e3:.0f} kJ, "
    f"to the ions {to_species[1] / 1e3:.0f} kJ, thermalised {thermalised / 1e3:.1f} kJ"
)
print(
    f"electron heating peaks at {time[electrons.argmax()]:.3f} s with {electrons.max() / 1e3:.0f} kW"
)
print(
    f"ion heating peaks at {time[ions.argmax()]:.3f} s with {ions.max() / 1e3:.0f} kW"
)
print(f"the ions get more than the electrons from {crossing:.2f} s on")

The electrons are heated at once: their heating peaks at the end of the blip and then decays with the fast ions' energy. The ion heating is small at first, since the 1 MeV ions are far above the critical energy, and grows as they slow down; it overtakes the electron heating about half a second after the blip, peaks just before the central ions thermalise, and ends there. Every joule is accounted for: the electrons get 102 kJ, the ions 59 kJ and the thermal ions the 3 kJ left at $1.5\,T_i$, together the 165 kJ of the blip.

## Square-wave modulation

A beam switched on and off with a square wave modulates the heating and the fast ions. Their modulation lags the beam: `response.transfer_function(frequency)` gives the complex gain $H_Q(f) = \int K_Q e^{-2\pi i f\tau} \mathrm{d}\tau / \int K_Q \mathrm{d}\tau$ of each moment, the modulation relative to the source's, from the same kernels. Here it is checked against the fundamental of the square-wave response, starting from the mean power and leaving out the first second:

In [ ]:
names = {
    "stored_energy": "fast-ion energy",
    "driven_current": "driven current",
    "electrons": "electron heating",
    "ions": "ion heating",
}


def moments(data):
    return {
        "stored_energy": data.stored_energy,
        "driven_current": data.driven_current,
        "electrons": data.total_heating.sel(species="electrons"),
        "ions": data.total_heating.sel(species="ions"),
    }


frequency = np.geomspace(0.2, 50.0, 200)
gain = moments(response.transfer_function(frequency))
delay = {k: -np.degrees(np.unwrap(np.angle(v.values))) for k, v in gain.items()}

dt = 5e-4
time = np.arange(0.0, 3.0, dt)
late = time >= 1.0
measured = {k: [] for k in names}
for f in (0.5, 1, 2, 5, 10, 20):
    square = response.evolve(
        time,
        lambda t, f=f: np.where((t * f) % 1.0 < 0.5, power, 0.0),
        initial_power=0.5 * power,
        reference_power=power,
    )
    phase = np.exp(-2j * np.pi * f * time[late])
    theory = moments(response.transfer_function(f))
    for name, values in moments(square).items():
        # the source's fundamental is (2/pi) sin(2 pi f t): a phase of -90 degrees
        lag = -np.degrees(np.angle(np.mean(values.values[late] * phase))) - 90.0
        expected = -np.degrees(np.angle(complex(theory[name][0])))
        assert abs((lag - expected + 180.0) % 360.0 - 180.0) < 0.5
        lag += 360.0 * np.round((np.interp(f, frequency, delay[name]) - lag) / 360.0)
        measured[name].append((f, lag))

smooth = ["stored_energy", "driven_current", "electrons"]
curves = xr.Dataset(
    {
        "gain": (("quantity", "frequency"), np.abs([v.values for v in gain.values()])),
        "delay": (("moment", "frequency"), [delay[k] for k in smooth]),
    },
    coords={
        "quantity": list(names.values()),
        "moment": [names[k] for k in smooth],
        "frequency": (
            "frequency",
            frequency,
            {"long_name": "frequency", "units": "Hz"},
        ),
    },
)
curves.gain.attrs.update(long_name="modulation gain", units="1")
curves.delay.attrs.update(long_name="phase delay", units="deg")
with plasma_plots.figure(2, 1, sharex=True, figsize=(6.5, 6.5)) as fig:
    curves.gain.plasma.plot.profiles(
        x="frequency", over="quantity", logx=True, logy=True, title="", ax=fig[0]
    )
    curves.delay.plasma.plot.profiles(
        x="frequency", over="moment", logx=True, title="", ax=fig[1]
    )
    for name in smooth:
        f, lag = np.transpose(measured[name])
        label = "square wave" if name == smooth[0] else None
        fig[1].plot(f, lag, "o", color="0.3", markersize=4, label=label)
    fig[1].legend(fontsize="small")

for name in names:
    print(
        f"{names[name]:>16}: delay {np.interp(1.0, frequency, delay[name]):4.0f} deg at 1 Hz, "
        f"gain {np.interp(1.0, frequency, np.abs(gain[name].values)):.2f} at 1 Hz "
        f"and {np.interp(10.0, frequency, np.abs(gain[name].values)):.3f} at 10 Hz"
    )

The phases of the square-wave runs agree with the transfer function to 0.5 degree, for all four moments. The fast-ion energy, the current and the electron heating behave like low-pass filters: their delay rises from about 20 degrees at 0.2 Hz to 60-85 degrees at 1 Hz and approaches 90 degrees above 10 Hz, and above a few hertz their gain falls as $1/f$. The small ripples on the curves come from the end of the slowing down, at about the same age for all central ions. The ion heating comes mostly late in the slowing down, so its kernel ends abruptly there: its gain falls fastest, with ripples, and its phase winds up by several turns (not drawn), so it has no useful delay. Above a few hertz the beam heats the ions almost steadily.

What the model leaves out, as `slow_down`: the orbit width, losses and charge exchange, and any change of the plasma by the beam (the heating does not change $T_e$ here, the response is linear). The source waveform is held constant over each time step; the response is exact for such a waveform.